# Multi-agent Matching Experiment

3エージェントの検出シミュレーションを生成し、RoCo-style、Pairwise RRWM、k-wise RRWM、Partial OT の4手法を比較します。

In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def find_repo_root(start: Path):
    """現在位置から親方向へ、src/simulation.py を持つ repo root を探す。"""
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / "src" / "simulation.py").exists():
            return path
    return None


repo_dir = find_repo_root(Path.cwd())

# VS Code から Colab kernel を使っている場合は /content にいるため、public repo を clone する。
if repo_dir is None and Path("/content").exists():
    repo_dir = Path("/content/Multi-agent-Matching")
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/furuyuu/Multi-agent-Matching.git", str(repo_dir)],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(repo_dir), "pull"], check=False)

if repo_dir is None or not (repo_dir / "src" / "simulation.py").exists():
    raise FileNotFoundError("Could not find repository root. Check your notebook kernel location.")

os.chdir(repo_dir)
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print("repo_dir:", Path.cwd())

## Imports

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from src.simulation import (
    count_detection_labels,
    generate_detections,
    generate_true_scene,
)
from src.visualization import (
    plot_detected_scene,
    plot_pairwise_matching_result,
    plot_true_scene,
)
from src.graph_matching import (
    build_method_comparison_dataframe,
    KWISE_PARAMS,
    PARTIAL_OT_PARAMS,
    ROCO_PARAMS,
    RRWM_PARAMS,
    evaluate_kwise_matching_3agents,
    kwise_rrwm_matching_3agents,
    kwise_to_pairwise_matches,
    print_kwise_matches_with_true_ids,
    print_matches_with_true_ids,
    print_method_comparison_table,
    run_pairwise_rrwm_all_pairs,
    run_roco_all_pairs,
)
from src.roco_iterative import (
    ROCO_ITERATIVE_PARAMS,
    build_estimated_object_position_dataframe,
    build_agent_pose_error_dataframe,
    build_object_pose_error_dataframe,
    build_pose_error_history_dataframe,
    build_pose_error_summary_dataframe,
    evaluate_iterative_roco_pose_error_history,
    evaluate_iterative_roco_pose_errors,
    build_iterative_matching_dataframe,
    run_iterative_pairwise_rrwm_pose_adjustment,
    run_iterative_partial_ot_pose_adjustment,
    run_iterative_kwise_rrwm_pose_adjustment,
    build_roco_iteration_dataframe,
    run_iterative_roco_pose_adjustment,
)

agent_pairs = [(0, 1), (0, 2), (1, 2)]

## Parameters

In [ ]:
# ここを変更すると、notebook 側で各手法のハイパーパラメータを調整できます。
# src.graph_matching の *_PARAMS はデフォルト値として使い、実験では copy した辞書を渡します。

roco_params = ROCO_PARAMS.copy()
roco_params.update(
    {
        "tau2": 6.0,
        "tau1": 0.3,
        "lambda_dist": 1.0,
        "neighbor_radius": 15.0,
    }
)

rrwm_params = RRWM_PARAMS.copy()
rrwm_params.update(
    {
        "candidate_radius": 6.0,
        "unary_weight": 2.0,
        "pairwise_weight": 1.0,
        "sigma_pos": 2.0,
        "sigma_yaw_deg": 15.0,
        "sigma_edge": 1.5,
        "max_iter": 10000,
        "alpha": 0.2,
        "beta": 30.0,
        "score_threshold": 0.02,
    }
)

kwise_params = KWISE_PARAMS.copy()
kwise_params.update(
    {
        "candidate_radius": 6.0,
        "unary_weight": 2.0,
        "pairwise_weight": 1.0,
        "sigma_pos": 2.0,
        "sigma_yaw_deg": 15.0,
        "sigma_edge": 1.5,
        "max_iter": 10000,
        "alpha": 0.2,
        "beta": 30.0,
        "score_threshold": 0.02,
    }
)

partial_ot_params = PARTIAL_OT_PARAMS.copy()
partial_ot_params.update(
    {
        "epsilon": 8.0,
        "epsilon_decay": 0.93,
        "beta2": 0.8,
        "outer_iter": 30,
        "inner_iter": 100,
        "tol": 1e-5,
        "match_score_threshold": 1e-4,
    }
)

print("roco_params:", roco_params)
print("rrwm_params:", rrwm_params)
print("kwise_params:", kwise_params)
print("partial_ot_params:", partial_ot_params)

roco_iterative_params = ROCO_ITERATIVE_PARAMS.copy()
roco_iterative_params.update(
    {
        "max_iter": 10,
        "damping": 0.8,
        "pose_tol": 1e-3,
        "min_matches_for_pose": 2,
    }
)

print("roco_iterative_params:", roco_iterative_params)

## Generate Simulation

In [ ]:
def get_output_dir(repo_root: Path, experiment_seed: int):
    """research/repo root の results 以下に、実行時刻つきフォルダを作って返します。"""
    from datetime import datetime
    from zoneinfo import ZoneInfo

    repo_root = Path(repo_root).resolve()
    timestamp = datetime.now(ZoneInfo("Asia/Tokyo")).strftime("%Y%m%d_%H%M%S_JST")
    output_dir = repo_root / "results" / f"{timestamp}_seed_{experiment_seed}"
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir


# 以前の Colab notebook と同じように、1つの rng をシーン生成と検出生成で共有します。
experiment_seed = 7
output_dir = get_output_dir(repo_dir, experiment_seed)
rng = np.random.default_rng(experiment_seed)

print("experiment_seed:", experiment_seed)
print("output_dir:", output_dir)

agents_gt, objects_gt = generate_true_scene(
    num_agents=3,
    num_objects=12,
    rng=rng,
)

agent_pose_est, detections_by_agent = generate_detections(
    agents_gt,
    objects_gt,
    sensing_range=32.0,
    fov_deg=150.0,
    detection_prob=0.85,
    object_pos_noise_std=0.7,
    object_yaw_noise_std_deg=4.0,
    agent_pos_noise_std=1.0,
    agent_yaw_noise_std_deg=3.0,
    num_outliers_per_agent=2,
    rng=rng,
)

for agent_id, (total, inliers, outliers) in count_detection_labels(detections_by_agent).items():
    print(
        f"Agent {agent_id}: detections={total}, "
        f"inliers={inliers}, outliers={outliers}"
    )

In [ ]:
plot_true_scene(
    agents_gt,
    objects_gt,
    save_path=output_dir / f"True_scene_seed_{experiment_seed}.png",
)
plot_detected_scene(
    agents_gt,
    agent_pose_est,
    detections_by_agent,
    save_path=output_dir / f"Observed_scene_seed_{experiment_seed}.png",
)

## Run Matching Methods

In [ ]:
roco_results = run_roco_all_pairs(
    detections_by_agent,
    agent_pairs,
    roco_params,
)

rrwm_results = run_pairwise_rrwm_all_pairs(
    detections_by_agent,
    agent_pairs,
    rrwm_params,
)

dets_0 = detections_by_agent[0]
dets_1 = detections_by_agent[1]
dets_2 = detections_by_agent[2]

kwise_matches, kwise_x, kwise_K, kwise_candidates = kwise_rrwm_matching_3agents(
    dets_0,
    dets_1,
    dets_2,
    **kwise_params,
)

kwise_eval = evaluate_kwise_matching_3agents(
    dets_0,
    dets_1,
    dets_2,
    kwise_matches,
)

print("k-wise evaluation:", kwise_eval)

## Matching Details

In [ ]:
for i, j in agent_pairs:
    print_matches_with_true_ids(
        f"RoCo-style matching: Agent {i} - Agent {j}",
        detections_by_agent[i],
        detections_by_agent[j],
        roco_results[(i, j)]["matches"],
    )
    print("evaluation:", roco_results[(i, j)]["evaluation"])

for i, j in agent_pairs:
    print_matches_with_true_ids(
        f"Pairwise RRWM matching: Agent {i} - Agent {j}",
        detections_by_agent[i],
        detections_by_agent[j],
        rrwm_results[(i, j)]["matches"],
    )
    print("evaluation:", rrwm_results[(i, j)]["evaluation"])

print_kwise_matches_with_true_ids(
    "k-wise MGM style RRWM matching: Agent 0 - 1 - 2",
    dets_0,
    dets_1,
    dets_2,
    kwise_matches,
)
print("evaluation:", kwise_eval)

## Comparison Table

In [ ]:
comparison_df = build_method_comparison_dataframe(
    agent_pairs=agent_pairs,
    roco_results=roco_results,
    rrwm_results=rrwm_results,
    kwise_matches=kwise_matches,
    detections_by_agent=detections_by_agent,
)
comparison_df.to_csv(output_dir / "matching_comparison.csv", index=False)


comparison_df.style.format(
    {
        "precision": "{:.3f}",
        "recall": "{:.3f}",
    }
).hide(axis="index")

## Matching Visualization

In [ ]:
for i, j in agent_pairs:
    plot_pairwise_matching_result(
        detections_by_agent[i],
        detections_by_agent[j],
        roco_results[(i, j)]["matches"],
        agent_i=i,
        agent_j=j,
        title=f"RoCo-style matching result: Agent {i} - Agent {j}",
        params=roco_params,
        save_path=output_dir / f"RoCo_{i}{j}_seed_{experiment_seed}.png",
    )

In [ ]:
for i, j in agent_pairs:
    plot_pairwise_matching_result(
        detections_by_agent[i],
        detections_by_agent[j],
        rrwm_results[(i, j)]["matches"],
        agent_i=i,
        agent_j=j,
        title=f"Pairwise RRWM matching result: Agent {i} - Agent {j}",
        params=rrwm_params,
        save_path=output_dir / f"Pairwise_{i}{j}_seed_{experiment_seed}.png",
    )

In [ ]:
for i, j in agent_pairs:
    pairwise_from_kwise = kwise_to_pairwise_matches(
        kwise_matches,
        pair=(i, j),
    )
    plot_pairwise_matching_result(
        detections_by_agent[i],
        detections_by_agent[j],
        pairwise_from_kwise,
        agent_i=i,
        agent_j=j,
        title=f"k-wise MGM result shown as pairwise: Agent {i} - Agent {j}",
        params=kwise_params,
        save_path=output_dir / f"kwise_{i}{j}_seed_{experiment_seed}.png",
    )

## Iterative Pose Adjustment

ここからは既存の matching-only 比較とは別に、matching と pose adjustment を交互に実行します。RoCo、Pairwise RRWM、k-wise RRWM を同じ初期条件で比較します。

In [ ]:
iterative_roco_result = run_iterative_roco_pose_adjustment(
    detections_by_agent=detections_by_agent,
    agent_pose_est=agent_pose_est,
    agent_pairs=agent_pairs,
    roco_params=roco_params,
    anchor_agent_id=0,
    **roco_iterative_params,
)

iterative_pairwise_rrwm_result = run_iterative_pairwise_rrwm_pose_adjustment(
    detections_by_agent=detections_by_agent,
    agent_pose_est=agent_pose_est,
    agent_pairs=agent_pairs,
    rrwm_params=rrwm_params,
    anchor_agent_id=0,
    **roco_iterative_params,
)

iterative_kwise_rrwm_result = run_iterative_kwise_rrwm_pose_adjustment(
    detections_by_agent=detections_by_agent,
    agent_pose_est=agent_pose_est,
    agent_pairs=agent_pairs,
    kwise_params=kwise_params,
    anchor_agent_id=0,
    **roco_iterative_params,
)

iterative_partial_ot_result = run_iterative_partial_ot_pose_adjustment(
    detections_by_agent=detections_by_agent,
    agent_pose_est=agent_pose_est,
    agent_pairs=agent_pairs,
    partial_ot_params=partial_ot_params,
    anchor_agent_id=0,
    **roco_iterative_params,
)

iterative_results = {
    "RoCo": iterative_roco_result,
    "Pairwise RRWM": iterative_pairwise_rrwm_result,
    "k-wise RRWM": iterative_kwise_rrwm_result,
    "Partial OT": iterative_partial_ot_result,
}

method_file_labels = {
    "RoCo": "roco",
    "Pairwise RRWM": "pairwise_rrwm",
    "k-wise RRWM": "kwise_rrwm",
    "Partial OT": "partial_ot",
}

iterative_matching_df = pd.concat(
    [build_iterative_matching_dataframe(result) for result in iterative_results.values()],
    ignore_index=True,
)
iterative_matching_df.to_csv(output_dir / "iterative_matching_history.csv", index=False)

iterative_matching_df.style.format(
    {
        "max_pose_delta": "{:.4f}",
        "precision": "{:.3f}",
        "recall": "{:.3f}",
    }
).hide(axis="index")

In [ ]:
print("Initial agent poses")
for agent_id, pose in agent_pose_est.items():
    print(f"  A{agent_id}: x={pose[0]:.3f}, y={pose[1]:.3f}, theta={np.rad2deg(pose[2]):.3f} deg")

for method_name, result in iterative_results.items():
    print(f"\nAdjusted agent poses ({method_name})")
    for agent_id, pose in result["agent_poses"].items():
        print(f"  A{agent_id}: x={pose[0]:.3f}, y={pose[1]:.3f}, theta={np.rad2deg(pose[2]):.3f} deg")
    print("Estimated object positions:", len(result["object_positions"]))

In [ ]:
estimated_object_dfs = {}
for method_name, result in iterative_results.items():
    method_label = method_file_labels[method_name]
    estimated_object_dfs[method_name] = build_estimated_object_position_dataframe(result)
    estimated_object_dfs[method_name].to_csv(
        output_dir / f"estimated_object_positions_{method_label}.csv",
        index=False,
    )

estimated_object_df = estimated_object_dfs["RoCo"]
estimated_object_df.style.format(
    {
        "x_est": "{:.3f}",
        "y_est": "{:.3f}",
    }
).hide(axis="index")

In [ ]:
pose_error_results = {
    method_name: evaluate_iterative_roco_pose_errors(result, agents_gt, objects_gt)
    for method_name, result in iterative_results.items()
}
pose_error_result = pose_error_results["RoCo"]

pose_error_history_by_method = {
    method_name: evaluate_iterative_roco_pose_error_history(result, agents_gt, objects_gt)
    for method_name, result in iterative_results.items()
}

pose_error_history_df = pd.concat(
    [
        build_pose_error_history_dataframe(pose_error_history)
        for pose_error_history in pose_error_history_by_method.values()
    ],
    ignore_index=True,
)
pose_error_history_df.to_csv(output_dir / "pose_error_history.csv", index=False)

display(
    pose_error_history_df.style.format(
        {
            "max_pose_delta": "{:.4f}",
            "mean_position_error": "{:.3f}",
            "rmse_position_error": "{:.3f}",
            "mean_yaw_error": "{:.4f}",
            "mean_yaw_error_deg": "{:.3f}",
            "mean_pose_error": "{:.3f}",
        }
    ).hide(axis="index")
)

pose_error_summary_df = pd.concat(
    [
        build_pose_error_summary_dataframe(error_result).assign(method=method_name)
        for method_name, error_result in pose_error_results.items()
    ],
    ignore_index=True,
)
pose_error_summary_df.to_csv(output_dir / "pose_error_summary.csv", index=False)

display(
    pose_error_summary_df.style.format(
        {
            "mean_position_error": "{:.3f}",
            "rmse_position_error": "{:.3f}",
            "mean_yaw_error": "{:.4f}",
            "mean_yaw_error_deg": "{:.3f}",
            "mean_pose_error": "{:.3f}",
        }
    ).hide(axis="index")
)


In [ ]:
agent_pose_error_dfs = {}
object_pose_error_dfs = {}
for method_name, error_result in pose_error_results.items():
    method_label = method_file_labels[method_name]
    agent_pose_error_dfs[method_name] = build_agent_pose_error_dataframe(error_result)
    object_pose_error_dfs[method_name] = build_object_pose_error_dataframe(error_result)
    agent_pose_error_dfs[method_name].to_csv(
        output_dir / f"agent_pose_errors_{method_label}.csv",
        index=False,
    )
    object_pose_error_dfs[method_name].to_csv(
        output_dir / f"object_pose_errors_{method_label}.csv",
        index=False,
    )

agent_pose_error_df = agent_pose_error_dfs["RoCo"]
object_pose_error_df = object_pose_error_dfs["RoCo"]

display(
    agent_pose_error_df.style.format(
        {
            "x_est": "{:.3f}",
            "y_est": "{:.3f}",
            "theta_est": "{:.4f}",
            "x_true": "{:.3f}",
            "y_true": "{:.3f}",
            "theta_true": "{:.4f}",
            "position_error": "{:.3f}",
            "yaw_error": "{:.4f}",
            "yaw_error_deg": "{:.3f}",
            "pose_error": "{:.3f}",
        }
    ).hide(axis="index")
)
display(
    object_pose_error_df.style.format(
        {
            "x_est": "{:.3f}",
            "y_est": "{:.3f}",
            "theta_est": "{:.4f}",
            "x_true": "{:.3f}",
            "y_true": "{:.3f}",
            "theta_true": "{:.4f}",
            "position_error": "{:.3f}",
            "yaw_error": "{:.4f}",
            "yaw_error_deg": "{:.3f}",
            "pose_error": "{:.3f}",
        }
    ).hide(axis="index")
)


In [ ]:
iterative_plot_labels = {
    "RoCo": "Iterative_RoCo",
    "Pairwise RRWM": "Iterative_Pairwise_RRWM",
    "k-wise RRWM": "Iterative_kwise_RRWM",
    "Partial OT": "Iterative_Partial_OT",
}

for method_name, result in iterative_results.items():
    adjusted_detections_by_agent = result["detections_by_agent"]
    adjusted_agent_pose_est = result["agent_poses"]
    plot_label = iterative_plot_labels[method_name]

    plot_detected_scene(
        agents_gt,
        adjusted_agent_pose_est,
        adjusted_detections_by_agent,
        save_path=output_dir / f"{plot_label}_adjusted_scene_seed_{experiment_seed}.png",
    )

    for i, j in agent_pairs:
        plot_pairwise_matching_result(
            adjusted_detections_by_agent[i],
            adjusted_detections_by_agent[j],
            result["pairwise_matches"][(i, j)],
            agent_i=i,
            agent_j=j,
            title=f"{method_name} iterative result: Agent {i} - Agent {j}",
            params={**roco_iterative_params},
            objects_gt=objects_gt,
            save_path=output_dir / f"{plot_label}_{i}{j}_seed_{experiment_seed}.png",
        )

In [ ]:
from html import escape


def build_experiment_report(output_dir: Path):
    """CSV と画像をまとめて確認できる HTML レポートを保存します。"""

    output_dir = Path(output_dir)
    csv_paths = sorted(output_dir.glob("*.csv"))
    image_paths = sorted(output_dir.glob("*.png"))

    csv_sections = []
    for csv_path in csv_paths:
        df = pd.read_csv(csv_path)
        table_html = df.to_html(
            index=False,
            classes="data-table",
            border=0,
            float_format=lambda value: f"{value:.4f}",
        )
        csv_sections.append(
            f"""
            <section>
              <div class=\"section-header\">
                <h2>{escape(csv_path.stem)}</h2>
                <a href=\"{escape(csv_path.name)}\">CSV</a>
              </div>
              <div class=\"table-wrap\">{table_html}</div>
            </section>
            """
        )

    image_cards = []
    for image_path in image_paths:
        image_cards.append(
            f"""
            <a class=\"image-card\" href=\"{escape(image_path.name)}\">
              <img src=\"{escape(image_path.name)}\" alt=\"{escape(image_path.stem)}\">
              <span>{escape(image_path.stem)}</span>
            </a>
            """
        )

    report_html = f"""
    <!doctype html>
    <html lang=\"ja\">
    <head>
      <meta charset=\"utf-8\">
      <meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">
      <title>Experiment Summary - {escape(output_dir.name)}</title>
      <style>
        body {{
          margin: 0;
          color: #202124;
          background: #f7f7f4;
          font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
        }}
        main {{ max-width: 1200px; margin: 0 auto; padding: 32px 20px 56px; }}
        h1 {{ margin: 0 0 6px; font-size: 28px; font-weight: 700; }}
        h2 {{ margin: 0; font-size: 18px; }}
        .meta {{ margin: 0 0 28px; color: #5f6368; }}
        section {{ margin-top: 28px; }}
        .section-header {{ display: flex; align-items: center; justify-content: space-between; gap: 16px; margin-bottom: 10px; }}
        .section-header a {{ color: #0b57d0; text-decoration: none; font-size: 14px; }}
        .image-grid {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(260px, 1fr)); gap: 14px; }}
        .image-card {{ display: block; overflow: hidden; border: 1px solid #dadce0; border-radius: 8px; background: white; color: inherit; text-decoration: none; }}
        .image-card img {{ display: block; width: 100%; height: 190px; object-fit: contain; background: #fff; border-bottom: 1px solid #eceff1; }}
        .image-card span {{ display: block; padding: 9px 10px; font-size: 13px; color: #3c4043; overflow-wrap: anywhere; }}
        .table-wrap {{ overflow-x: auto; border: 1px solid #dadce0; border-radius: 8px; background: white; }}
        table.data-table {{ border-collapse: collapse; width: 100%; font-size: 13px; }}
        .data-table th, .data-table td {{ padding: 7px 9px; border-bottom: 1px solid #eceff1; text-align: right; white-space: nowrap; }}
        .data-table th {{ position: sticky; top: 0; background: #eef2f7; color: #202124; font-weight: 650; }}
        .data-table th:first-child, .data-table td:first-child {{ text-align: left; }}
        .data-table tr:nth-child(even) td {{ background: #fafafa; }}
      </style>
    </head>
    <body>
      <main>
        <h1>Experiment Summary</h1>
        <p class=\"meta\">{escape(output_dir.name)}</p>
        <section>
          <div class=\"section-header\"><h2>Figures</h2></div>
          <div class=\"image-grid\">{''.join(image_cards)}</div>
        </section>
        {''.join(csv_sections)}
      </main>
    </body>
    </html>
    """

    report_path = output_dir / "summary.html"
    report_path.write_text(report_html, encoding="utf-8")
    return report_path


report_path = build_experiment_report(output_dir)
print("report_path:", report_path)